In [1]:
import sys
sys.path.append('../') 
import DirectStiffness as DS

In [5]:
DS.Structure()

2026-05-14 09:48:27.948 | DEBUG    | DirectStiffness.utils:__init__:13 - Config init called
2026-05-14 09:48:27.949 | DEBUG    | DirectStiffness.utils:_select_device:34 - NVIDIA GPU detected : NVIDIA GeForce RTX 3060 Laptop GPU
2026-05-14 09:48:27.949 | DEBUG    | DirectStiffness.utils:__init__:22 - Using cuda as the default device.


In [6]:
cfg = DS.Config()

2026-05-14 09:48:27.956 | DEBUG    | DirectStiffness.utils:__init__:13 - Config init called
2026-05-14 09:48:27.956 | DEBUG    | DirectStiffness.utils:_select_device:34 - NVIDIA GPU detected : NVIDIA GeForce RTX 3060 Laptop GPU
2026-05-14 09:48:27.957 | DEBUG    | DirectStiffness.utils:__init__:22 - Using cuda as the default device.


In [2]:
problem = {
    "materials": {
        "E": [2e8, 2e8],
        "I": [4e-6, 4e-6],
        "A": [0.0, 0.0],
    },
    "nodes": [
        [0.0, 0.0],
        [5.0, 0.0],
        [10.0, 0.0],
    ],
    "connectivity": [
        [0, 1],
        [1, 2],
    ],
    "free_dofs": [4, 5],
    "loads": {
        4: -1.0,
    },
}

In [3]:
structure = DS.Structure()
structure.read_variables(problem)

2026-05-14 10:39:30.786 | DEBUG    | DirectStiffness.utils:__init__:13 - Config init called
2026-05-14 10:39:30.905 | DEBUG    | DirectStiffness.utils:_select_device:34 - NVIDIA GPU detected : NVIDIA GeForce RTX 3060 Laptop GPU
2026-05-14 10:39:30.906 | DEBUG    | DirectStiffness.utils:__init__:22 - Using cuda as the default device.


Node ID: 0
Node ID: 1
Node ID: 2
Elem ID: 0
Elem ID: 1


In [ ]:
structure.nodes[0] ## __repr__

Node(id=0, x=0.0, y=0.0)

In [ ]:

print(structure.nodes[0]) ## __str__

Node ID: 0


In [9]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)
np.set_printoptions(linewidth=np.inf)
# Define the default float type you want to use
float_type = np.float64


In [10]:
# E_M=[2*10**8,2*10^8,2*10^8,2*10^8,2*10^8]                             # modulus of Elasticity matrix 
# dep=0.1
# base=.05
# I_M=[1*10^(-5),1*10^(-5),1*10^(-5),1*10^(-5),1*10^(-5)]    # moment of inertia matrix
# A_M=[0.01,0.01,0.01,0.01,0.01]    # cross-section area matrix
# node=[[0, 0],[0, 3],[0, 6],[0, 8]] # nodal coordinates
# conn=[[0, 1],[1, 2],[2, 3]]         #element connection matrix       
# NN= len(node)        #number of nodes
# NE= len(conn)        #number of elements
# ndof= 3*NN              #total dof (axial, shear, moment)
# fdof=[3, 4, 5, 6, 7, 8, 11]         #free dof

In [11]:
E_M = [2e8, 2e8]                       # modulus of Elasticity matrix 
dep=0.1
base=.05
I_M = [4e-6, 4e-6]   # moment of inertia matrix
A_M=[0.0,0.0]    # cross-section area matrix
node = np.array([[0.0, 0.0], [5.0, 0.0], [10.0, 0.0]]) # nodal coordinates
conn = np.array([[0, 1], [1, 2]])          #element connection matrix       
fdof=[4, 5]         #free dof

In [12]:
NN= len(node)        #number of nodes
NE= len(conn)        #number of elements
ndof= 3*NN              #total dof (axial, shear, moment)

In [13]:
# Formation of Displacement Matrix
d = np.zeros([ndof, 1], dtype=float_type) 
f = np.zeros([ndof, 1], dtype=float_type)
K = np.zeros([ndof, ndof], dtype=float_type)     #size of assembled stiffness matrix is ndof x ndof
f[4]=-1.0
#f(5)=-10;f(11)=-10;%empty load vector
#f(2)=-4;f(3)=-2.66666666666667;f(5)=-4-2-10;f(6)=+2.66666666666667-0.666666666666667;f(8)=-2;f(9)=0.666666666666667;

In [14]:
f, conn

(array([[ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [-1.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.]]),
 array([[0, 1],
        [1, 2]]))

In [15]:
# Formation of Stiffness Matrix
        
for e in range (0,NE):              #loop over all elements
    print("----------------------")
    print("elem",e)
    n1=conn[e][0]       #ID of first node of each element
    print("n1",n1)
    n2=conn[e][1]       #ID of second node of each element
    print("n2",n2)

    x1=node[n1][0]      #x coordinate of node n1
    print("x1",x1)
    x2=node[n2][0]      #x coordinate of node n2
    print("x2",x2)
    y1=node[n1][1]      #y coordinate of node n1
    print("y1",y1)
    y2=node[n2][1]      #y coordinate of node n2
    print("y2",y2)

    L=((x2-x1)**2+(y2-y1)**2)**0.5 # perform sqrt using **0.5 ## sqrt((x2-x1)^2+(y2-y1)^2);
    print("L elem", L)
    theta=np.arctan2((y2-y1),(x2-x1))
    print("theta", np.degrees(theta))
    c=np.cos(theta)
    s=np.sin(theta)
    E=E_M[e]
    I=I_M[e]
    A=A_M[e]
    K_W_R=  np.array([
    [E*A/L,      0,            0,           -E*A/L,      0,            0],
    [0,      12*E*I/L**3,   6*E*I/L**2,     0,      -12*E*I/L**3,   6*E*I/L**2],
    [0,       6*E*I/L**2,   4*E*I/L,        0,      -6*E*I/L**2,    2*E*I/L],
    [-E*A/L,     0,            0,            E*A/L,      0,            0],
    [0,     -12*E*I/L**3,  -6*E*I/L**2,    0,       12*E*I/L**3,  -6*E*I/L**2],
    [0,       6*E*I/L**2,   2*E*I/L,        0,      -6*E*I/L**2,    4*E*I/L]
    ])
    if (y2-y1)==0: # if not inclined don't multiply and waste wall time
        KE=K_W_R
    else:
        R=np.array([[c, s, 0, 0, 0, 0],
                    [-s, c, 0, 0, 0, 0],
                    [ 0, 0, 1, 0, 0, 0],
                    [ 0, 0, 0, c, s, 0],
                    [ 0, 0, 0, -s, c, 0],
                    [ 0, 0, 0, 0, 0, 1]])
        KE=np.matmul(np.transpose(R),(K_W_R),R)

    # Assembly of Stiffness Matrix
    sctr=[3*n1, 3*n1+1, 3*n1+2, 3*n2, 3*n2+1, 3*n2+2] # sctr
    # print(np.ix_(sctr, sctr))
    K[np.ix_(sctr, sctr)] += KE
    # print(np.array_str(K, precision=2, suppress_small=True))
    

----------------------
elem 0
n1 0
n2 1
x1 0.0
x2 5.0
y1 0.0
y2 0.0
L elem 5.0
theta 0.0
----------------------
elem 1
n1 1
n2 2
x1 5.0
x2 10.0
y1 0.0
y2 0.0
L elem 5.0
theta 0.0


In [16]:
np.shape(KE),np.shape(K)

((6, 6), (9, 9))

In [17]:
# Nodal Displacements
# d[np.ix_(fdof)]=np.linalg.solve(K[np.ix_(fdof,fdof)],f[np.ix_(fdof)])
K_sub = K[np.ix_(fdof, fdof)]
f_sub = f[np.ix_(fdof)]

K_sub, f_sub

(array([[ 153.6,    0. ],
        [   0. , 1280. ]]),
 array([[-1.],
        [ 0.]]))

In [18]:
# 2. Solve the reduced system
d_sub = np.linalg.solve(K_sub, f_sub)

# 3. Assign the results back to the full displacement vector
d[fdof] = d_sub
d_sub

array([[-0.007],
       [ 0.   ]])

In [19]:
np.set_printoptions(linewidth=np.inf)
K

array([[   0. ,    0. ,    0. ,    0. ,    0. ,    0. ,    0. ,    0. ,    0. ],
       [   0. ,   76.8,  192. ,    0. ,  -76.8,  192. ,    0. ,    0. ,    0. ],
       [   0. ,  192. ,  640. ,    0. , -192. ,  320. ,    0. ,    0. ,    0. ],
       [   0. ,    0. ,    0. ,    0. ,    0. ,    0. ,    0. ,    0. ,    0. ],
       [   0. ,  -76.8, -192. ,    0. ,  153.6,    0. ,    0. ,  -76.8,  192. ],
       [   0. ,  192. ,  320. ,    0. ,    0. , 1280. ,    0. , -192. ,  320. ],
       [   0. ,    0. ,    0. ,    0. ,    0. ,    0. ,    0. ,    0. ,    0. ],
       [   0. ,    0. ,    0. ,    0. ,  -76.8, -192. ,    0. ,   76.8, -192. ],
       [   0. ,    0. ,    0. ,    0. ,  192. ,  320. ,    0. , -192. ,  640. ]])

In [20]:
d

array([[ 0.   ],
       [ 0.   ],
       [ 0.   ],
       [ 0.   ],
       [-0.007],
       [ 0.   ],
       [ 0.   ],
       [ 0.   ],
       [ 0.   ]])

In [21]:
f = -f + np.matmul(K,d)
f

array([[ 0.  ],
       [ 0.5 ],
       [ 1.25],
       [ 0.  ],
       [ 0.  ],
       [ 0.  ],
       [ 0.  ],
       [ 0.5 ],
       [-1.25]])

In [22]:
from prettytable import PrettyTable

In [23]:
t = PrettyTable(['Node', 'X-Disp', 'Y-disp', 'Rotation'])
for i in range(0,NN):
    t.add_row([i, d[3*i], d[3*i+1], d[3*i+2]])

print(t)


+------+--------+----------+----------+
| Node | X-Disp |  Y-disp  | Rotation |
+------+--------+----------+----------+
|  0   |  [0.]  |   [0.]   |   [0.]   |
|  1   |  [0.]  | [-0.007] |   [0.]   |
|  2   |  [0.]  |   [0.]   |   [0.]   |
+------+--------+----------+----------+


In [24]:
t = PrettyTable(['Node', 'X-Force', 'Y-Force', 'Z-Moment'])
for i in range(0,NN):
    t.add_row([i, f[3*i], f[3*i+1], f[3*i+2]])

print(t)


+------+---------+---------+----------+
| Node | X-Force | Y-Force | Z-Moment |
+------+---------+---------+----------+
|  0   |   [0.]  |  [0.5]  |  [1.25]  |
|  1   |   [0.]  |   [0.]  |   [0.]   |
|  2   |   [0.]  |  [0.5]  | [-1.25]  |
+------+---------+---------+----------+
